# Homework 14: Washout Predicts the Patient

**Computational Sensorimotor Control | Week 14 of 15**

**Due:** One week from lab session

In this homework you investigate how cerebellar degradation affects the washout and re-learning phases of motor adaptation. You run the two-stage model from Lab 14 at four levels of fast-process impairment and measure two clinically observable signatures: **spontaneous recovery** (during washout) and **savings** (during re-learning).

The central question: can you distinguish a mildly impaired patient from a healthy control using only the washout and re-learning phases — without seeing the initial adaptation?

**Deliverables:** Submit HW14_WashoutPredicts.ipynb with all cells executed, all three plots, and all written answers.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.figsize': (12, 4), 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Two-stage model from Lab 14 (provided)
def simulate_two_stage(perturbation, Af=0.85, As=0.998, eta_f=0.15, eta_s=0.03):
    N = len(perturbation)
    xf = 0.0; xs = 0.0
    error = np.zeros(N); xf_hist = np.zeros(N); xs_hist = np.zeros(N); xtotal_hist = np.zeros(N)
    for n in range(N):
        x_total = xf + xs
        e = perturbation[n] - x_total
        error[n] = e
        xf_hist[n] = xf; xs_hist[n] = xs; xtotal_hist[n] = x_total
        xf = Af * xf + eta_f * e
        xs = As * xs + eta_s * e
    return error, xf_hist, xs_hist, xtotal_hist

print('Setup complete.')

## Part 1: Run the Two-Stage Model at Four Degradation Levels (1 line)

Use the perturbation schedule: 20 baseline, 80 adaptation (perturbation = 1.0), 50 washout (perturbation = 0), and 60 re-adaptation (perturbation = 1.0).

Run the model for four values of $\eta_f$: 0.15 (healthy), 0.10 (mild), 0.05 (moderate), and 0.02 (severe). All other parameters stay at their defaults.

The starter code handles the loop — you fill in one line to run the model.

In [ ]:
n_base, n_adapt, n_wash, n_readapt = 20, 80, 50, 60
perturbation = np.concatenate([
    np.zeros(n_base), np.ones(n_adapt), np.zeros(n_wash), np.ones(n_readapt)
])
N_total = len(perturbation)

eta_f_vals = [0.15, 0.10, 0.05, 0.02]
labels = ['Healthy (0.15)', 'Mild (0.10)', 'Moderate (0.05)', 'Severe (0.02)']
colors = ['#2166ac', '#67a9cf', '#ef8a62', '#b2182b']

results = {}
for eta_f, label in zip(eta_f_vals, labels):
    ### YOUR CODE: run the two-stage model (1 line) ###
    # error, xf, xs, xtotal = simulate_two_stage(perturbation, eta_f=...)
    results[eta_f] = {'error': error, 'xf': xf, 'xs': xs, 'xtotal': xtotal}

# ── Plot: Full learning curves ──
fig, ax = plt.subplots(figsize=(12, 4))
for eta_f, label, col in zip(eta_f_vals, labels, colors):
    ax.plot(results[eta_f]['error'], color=col, lw=1.2, label=label)
ax.axvspan(0, n_base, alpha=0.05, color='gray')
ax.axvspan(n_base, n_base+n_adapt, alpha=0.05, color='blue')
ax.axvspan(n_base+n_adapt, n_base+n_adapt+n_wash, alpha=0.05, color='red')
ax.axvspan(n_base+n_adapt+n_wash, N_total, alpha=0.05, color='green')
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('Trial'); ax.set_ylabel('Error')
ax.set_title('Full adaptation cycle at four degradation levels')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

> **Q1:** Which degradation level shows the slowest initial adaptation? Is the adaptation rate change proportional to the $\eta_f$ reduction, or is the relationship nonlinear?

## Part 2: Spontaneous Recovery Magnitude (2 lines)

During washout, the fast process extinguishes and the slow process persists, producing a transient error rebound — spontaneous recovery. Measure the **peak negative error during washout** for each degradation level. This is the spontaneous recovery magnitude.

Plot spontaneous recovery magnitude vs $\eta_f$.

In [ ]:
wash_start = n_base + n_adapt
wash_end = wash_start + n_wash

sr_magnitude = []
for eta_f in eta_f_vals:
    wash_errors = results[eta_f]['error'][wash_start:wash_end]
    ### YOUR CODE: compute spontaneous recovery magnitude (1 line) ###
    # sr = np.min(wash_errors)  # most negative error during washout
    sr_magnitude.append(sr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel A: Washout zoom for all levels
ax = axes[0]
for eta_f, label, col in zip(eta_f_vals, labels, colors):
    ax.plot(np.arange(n_wash), results[eta_f]['error'][wash_start:wash_end],
            color=col, lw=1.5, label=label)
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlabel('Trials into washout'); ax.set_ylabel('Error')
ax.set_title('A: Washout phase'); ax.legend(fontsize=8)

# Panel B: SR magnitude vs eta_f
ax = axes[1]
### YOUR CODE: plot SR magnitude vs eta_f (1 line) ###
# ax.plot(eta_f_vals, sr_magnitude, 'ko-', ms=8, lw=2)
ax.set_xlabel('$\eta_f$ (fast learning rate)'); ax.set_ylabel('Spontaneous recovery (peak negative error)')
ax.set_title('B: Spontaneous recovery vs cerebellar integrity')
ax.invert_yaxis()  # more negative = larger SR, so invert for intuition
plt.tight_layout(); plt.show()

for eta_f, sr in zip(eta_f_vals, sr_magnitude):
    print(f'eta_f = {eta_f:.2f}: spontaneous recovery = {sr:.4f}')

> **Q2a:** Why does spontaneous recovery shrink as $\eta_f$ decreases?

> **Q2b:** At $\eta_f = 0.02$, is spontaneous recovery completely absent? If not, where does the residual come from?

## Part 3: Savings Magnitude (2 lines)

Savings = faster re-learning compared to initial learning. Measure savings as the **difference in mean error over the first 10 trials** of initial adaptation vs re-adaptation. Larger difference = more savings.

Plot savings vs $\eta_f$.

In [ ]:
readapt_start = wash_end

savings = []
for eta_f in eta_f_vals:
    init_err = results[eta_f]['error'][n_base:n_base+10]
    re_err = results[eta_f]['error'][readapt_start:readapt_start+10]
    ### YOUR CODE: compute savings as difference in mean error (1 line) ###
    # sav = np.mean(init_err) - np.mean(re_err)
    savings.append(sav)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel A: Initial vs re-learning for each level
ax = axes[0]
for eta_f, label, col in zip(eta_f_vals, labels, colors):
    init_err = results[eta_f]['error'][n_base:n_base+30]
    re_err = results[eta_f]['error'][readapt_start:readapt_start+30]
    ax.plot(np.arange(30), init_err, color=col, ls='-', lw=1.5, alpha=0.5)
    ax.plot(np.arange(30), re_err, color=col, ls='--', lw=1.5)
ax.set_xlabel('Trials into adaptation'); ax.set_ylabel('Error')
ax.set_title('A: Initial (solid) vs re-learning (dashed)')

# Panel B: Savings vs eta_f
ax = axes[1]
### YOUR CODE: plot savings vs eta_f (1 line) ###
# ax.plot(eta_f_vals, savings, 'ko-', ms=8, lw=2)
ax.set_xlabel('$\eta_f$ (fast learning rate)'); ax.set_ylabel('Savings (mean error difference)')
ax.set_title('B: Savings vs cerebellar integrity')
plt.tight_layout(); plt.show()

for eta_f, sav in zip(eta_f_vals, savings):
    print(f'eta_f = {eta_f:.2f}: savings = {sav:.4f}')

> **Q3a:** Does savings decrease monotonically with $\eta_f$, or is the relationship more complex? Why might savings be partially preserved even when the fast process is severely impaired?

> **Q3b:** Design an experiment (describe the trial schedule and what you would measure) that could distinguish a mildly impaired cerebellar patient ($\eta_f \approx 0.10$) from a healthy control, using only the washout and re-learning phases — without seeing the initial adaptation. Which metric (spontaneous recovery or savings) would be more diagnostic, and why?

## Summary

| Part | What you measured | Lines of code |
|------|------------------|---------------|
| 1 | Full adaptation at 4 degradation levels | 1 |
| 2 | Spontaneous recovery vs $\eta_f$ | 2 |
| 3 | Savings vs $\eta_f$ | 2 |